# The ScikitLearn.jl library

The Scikit-learn library is an open-source machine learning library developed for the Python programming language, the first version of which dates back to 2010. It implements many machine learning models, related to classification, regression, clustering or dimensionality reduction. These models include Support Vector Machines (SVM), decision trees, random forests, or k-means. It is currently one of the most widely used libraries in the field of machine learning, due to the large number of functionalities it offers as well as its ease of use, since it provides a uniform interface for training and using models. The documentation for this library is available at https://scikit-learn.org/stable/.

For Julia, the ScikitLearn.jl library implements this interface and the algorithms contained in the scikit-learn library, supporting both Julia's own models and those of the scikit-learn library. The latter is done by means of the PyCall.jl library, which allows code written in Python to be executed from Julia in a transparent way for the user, who only needs to have ScikitLearn.jl installed. Documentation for this library can be found at https://scikitlearnjl.readthedocs.io/en/latest/.

However, recently, some incompatibilities have been reported with some versions of the SSL library. To avoid potential compatibility issues between Julia, PyCall, and ScikitLearn, we will use a different library for this exercise.
The library we will use is MLJ (Machine Learning in Julia), which is not strictly a library but rather a framework that allows the use of various related libraries through a common interface.
As a result, the function names used to create and train models remain the same regardless of the specific models being used.
In the practical sessions of this course, in addition to ANNs, we will use the following models, available within the MLJ framework:

- Support Vector Machines (SVM)
- Decision trees
- kNN

In order to use these models, it is first necessary to install and import the library:

In [96]:
import Pkg;
#Pkg.add("MLJ")
#Pkg.add("MLJLIBSVMInterface")
#Pkg.build("PyCall"
#Pkg.add("NearestNeighborModels")
#Pkg.add("MLJDecisionTreeInterface")
#Pkg.add("LIBSVM")

using MLJ;
using Random
using LIBSVM

#----------------------
# Import utility functions
#----------------------
include("utils.jl")


ANNCrossValidation (generic function with 1 method)

Similarly, it is necessary to install the packages that contain the specific learning algorithms (e.g., LIBSVM, NearestNeighborModels, DecisionTree) as well as the packages that provide the interfaces between these algorithms and the MLJ framework (MLJLIBSVMInterface, MLJDecisionTreeInterface).
To import the models to be used, we can rely on the `MLJ.@load` macro. For example, the following lines import the three models mentioned above, which will be used in this course:

In [97]:
SVMClassifier = MLJ.@load SVC pkg=LIBSVM verbosity=0
kNNClassifier = MLJ.@load KNNClassifier pkg=NearestNeighborModels verbosity=0
DTClassifier = MLJ.@load DecisionTreeClassifier pkg=DecisionTree verbosity=0

MLJDecisionTreeInterface.DecisionTreeClassifier

As can be seen, each model is loaded from a different package. The `verbosity=0` option is simply used to suppress the output message that would otherwise be printed during the import.
This way, we define three functions to create each one of the three models. Each function receives as arguments the specific hyperparameters for the corresponding model.
Below are three examples, one for each type of model that will be used in these course exercises:

In [98]:
model = SVMClassifier(kernel=LIBSVM.Kernel.RadialBasis, cost=1.0, gamma=2.0, degree=Int32(3))

SVC(
  kernel = LIBSVM.Kernel.RadialBasis, 
  gamma = 2.0, 
  cost = 1.0, 
  cachesize = 200.0, 
  degree = 3, 
  coef0 = 0.0, 
  tolerance = 0.001, 
  shrinking = true)

In [99]:
model = DTClassifier(max_depth=4, rng=Random.MersenneTwister(1))

DecisionTreeClassifier(
  max_depth = 4, 
  min_samples_leaf = 1, 
  min_samples_split = 2, 
  min_purity_increase = 0.0, 
  n_subfeatures = 0, 
  post_prune = false, 
  merge_purity_threshold = 1.0, 
  display_depth = 5, 
  feature_importance = :impurity, 
  rng = MersenneTwister(1))

In [100]:
model = kNNClassifier(K=3)

KNNClassifier(
  K = 3, 
  algorithm = :kdtree, 
  metric = Euclidean(0.0), 
  leafsize = 10, 
  reorder = true, 
  weights = Uniform())

When creating a kNN model, the main hyperparameter is `K`, which defines the number of neighbors.
For decision trees, the main hyperparameter is `max_depth`, which sets the maximum depth of the tree.
In the case of decision trees, as shown earlier, there is also a parameter called `rng`. This parameter controls the randomness involved in a specific part of the model construction process.

Specifically, for decision trees, this randomness occurs during the selection of features used to split a node. The `DecisionTree` library uses a random number generator (RNG) for this step, which is updated with each call. As a result, different calls to the function (along with subsequent calls to `fit!`) may produce different models, even with the same data.

To control this randomness and make the process deterministic, it is advisable to provide a fixed integer value as the RNG seed, as shown in the previous example.
This ensures that creating a model with a given input-output dataset and a defined set of hyperparameters becomes a reproducible process.

In general, it is preferable to control randomness across the entire model development workflow (e.g., cross-validation, training/test splits) by setting a global random seed at the beginning.
However, for the purposes of these exercises, we will use the `rng` keyword specifically for the decision tree model.

SVMs have a more complex set of hyperparameters, which depend on the kernel function being used.  
First, the hyperparameter `C` controls the trade-off between the margin width and classification error. Lower values allow for more misclassifications (more tolerance), while higher values fit the model more tightly to the data.  
In MLJ, this parameter is passed using the keyword `cost` when calling `SVMClassifier`.

Additionally, it is necessary to specify which kernel to use. This is done using the keyword `kernel`, which can take one of the following values provided by the `LIBSVM` library:

- `LIBSVM.Kernel.Linear`
- `LIBSVM.Kernel.RadialBasis`
- `LIBSVM.Kernel.Sigmoid`
- `LIBSVM.Kernel.Polynomial`

Depending on the kernel selected, different additional hyperparameters are used:

- **Linear kernel**: Only requires `C` (via `cost`).
- **RBF (Radial Basis Function) kernel**: In addition to `C`, it uses `gamma`, which controls the influence of each support vector.
- **Sigmoid kernel**: Uses `C`, `gamma`, and `coef0`. This kernel behaves similarly to a neural network, where `gamma` and `coef0` influence the shape of the decision function.
- **Polynomial kernel**: Uses `C`, `degree` (the degree of the polynomial), `gamma`, and `coef0`.

Typical values for these hyperparameters include:  
`0.001`, `0.1`, `1`, `10`, `100`, `1000`.

The following table summarises the different hyperparameters, the kernels that use them, and typical values they may take.

Note that when calling the `SVMClassifier` function, the hyperparameters use the same names as listed here, except for `C`, which must be passed as `cost`, as shown in the previous example.

It is also important that the arguments passed to the function have the correct type, as required by the `LIBSVM` library. Otherwise, an error may occur.

To prevent this, it is recommended to explicitly cast each hyperparameter to the appropriate type when calling the function.

| Hyperparameter | Applicable Kernels                 | Typical Values                     | Required Type in LIBSVM |
|----------------|------------------------------------|------------------------------------|--------------------------|
| `cost` (`C`)   | Linear, RBF, Sigmoid, Polynomial   | 0.001, 0.1, 1, 10, 100, 1000       | `Float64`               |
| `gamma`        | RBF, Sigmoid, Polynomial           | 0.1, 0.01, 0.001, 0.0001           | `Float64`               |
| `coef0`        | Sigmoid, Polynomial                | 0, 1, 5, 10                        | `Int32`                 |
| `degree`       | Polynomial                         | 2, 3, 4, 5                         | `Float64`               |

Although the basic SVM model is inherently binary, the implementation provided in MLJ already supports multi-class classification.  
Therefore, it is not necessary to manually apply a one-vs-all strategy for multi-class problems.

Once a model has been created, it must be wrapped in a `machine` object. This object acts as a container that associates the model with the data and handles both training and prediction.  
It is a core concept in MLJ and simplifies model workflows by centralizing model fitting (`fit!`) and prediction (`predict`) logic.

A `machine` has three main components:

- **Model**: Specifies the algorithm to be used. It has been created earlier, without any data or learned state.
- **Data**: Provides the input features and target labels (if supervised).
- **Internal State**: Stores learned parameters after the model is trained.

To create a `machine`, you can use the `machine` function, passing in the model, the input features, and the target labels.  
Note that the input data must not be plain arrays. Instead, it should be converted to a supported table format such as `Tables.table`, `DataFrame`, or a `NamedTuple`.  
If your data is currently stored as arrays, as is the case in these exercises, the following line shows how to construct the machine:

# Dprecated

# Load Iris dataset

In [101]:
trainIdx, valIdx, testIdx = holdOut(10, 0.3, 0.3);
println("Train indices: ", trainIdx)
println("Validation indices: ", valIdx)
println("Test indices: ", testIdx)


#Load the data from previous notebooks
# 1. Load the database
using DelimitedFiles;
dataset = readdlm("./data/iris.data",',');

trainIdx, valIdx, testIdx = holdOut(size(dataset, 1), 0.3, 0.3)
println("Train indices: ", length(trainIdx))
println("Validation indices: ", length(valIdx))
println("Test indices: ", length(testIdx))
println("Dataset size: ", size(dataset))


# 1.1. Prepare the data
inputs = convert(Array{Float32,2}, dataset[:,1:4]);
targets = dataset[:,5];
classes = unique(targets);

println("targets \n $targets \n size$(size(targets))")
println("classes \n $classes \n num classes : $(length(classes))")

# # 1.2 One-hot encode targets
# targets_oh = oneHotEncoding(targets, classes)
# println("One-hot encoded targets size: ", size(targets_oh))


# 1.3 Create training, validation, and test sets 
train_inputs = inputs[trainIdx, :]
val_inputs   = inputs[valIdx, :]
test_inputs  = inputs[testIdx, :]

# train_targets = targets_oh[trainIdx, :]
# val_targets   = targets_oh[valIdx, :]
# test_targets  = targets_oh[testIdx, :]

println("Train set: inputs $(size(train_inputs)), targets $(size(train_targets))")
println("Validation set: inputs $(size(val_inputs)), targets $(size(val_targets))")
println("Test set: inputs $(size(test_inputs)), targets $(size(test_targets))")

println("Train set: type of  $(typeof(train_inputs)), targets: type of $(typeof(train_targets))")


Train indices: [2, 6, 8, 3]
Validation indices: [10, 7, 5]
Test indices: [9, 1, 4]
Train indices: 60
Validation indices: 45
Test indices: 45
Dataset size: (150, 5)
targets 
 Any["Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-versicolor", "Iris-versicolor", "Iris-versicolor", "Iris-versicolo

SVM : Note 
 - MLJ’s SVMClassifier expects a 1D vector of categorical labels

# DEPRECATED

In [102]:
using CategoricalArrays

# Convert targets (string labels) into a categorical vector
targets_cat = categorical(targets)
# Split into train/val/test
train_targets = targets_cat[trainIdx]
val_targets   = targets_cat[valIdx]
test_targets  = targets_cat[testIdx]
println("Targets  $(train_targets)")


Targets  CategoricalValue{String, UInt32}["Iris-virginica", "Iris-setosa", "Iris-setosa", "Iris-versicolor", "Iris-versicolor", "Iris-versicolor", "Iris-setosa", "Iris-virginica", "Iris-setosa", "Iris-versicolor", "Iris-virginica", "Iris-versicolor", "Iris-setosa", "Iris-setosa", "Iris-versicolor", "Iris-setosa", "Iris-versicolor", "Iris-setosa", "Iris-versicolor", "Iris-setosa", "Iris-versicolor", "Iris-setosa", "Iris-virginica", "Iris-versicolor", "Iris-virginica", "Iris-virginica", "Iris-virginica", "Iris-virginica", "Iris-versicolor", "Iris-virginica", "Iris-setosa", "Iris-versicolor", "Iris-setosa", "Iris-setosa", "Iris-setosa", "Iris-virginica", "Iris-versicolor", "Iris-versicolor", "Iris-setosa", "Iris-versicolor", "Iris-virginica", "Iris-versicolor", "Iris-virginica", "Iris-virginica", "Iris-versicolor", "Iris-virginica", "Iris-virginica", "Iris-virginica", "Iris-virginica", "Iris-setosa", "Iris-virginica", "Iris-setosa", "Iris-virginica", "Iris-setosa", "Iris-setosa", "Iris-vi

In [103]:
# define the model
model = SVMClassifier(kernel=LIBSVM.Kernel.RadialBasis, cost=1.0, gamma=2.0, degree=Int32(3))
# create the machine object
mach = machine(model, MLJ.table(train_inputs), categorical(train_targets))

untrained Machine; caches model-specific representations of data
  model: SVC(kernel = RadialBasis, …)
  args: 
    1:	Source @638 ⏎ Table{AbstractVector{Continuous}}
    2:	Source @315 ⏎ AbstractVector{Multiclass{3}}


As shown, the input matrix is converted into a table, and the target vector is converted into a categorical array, since this exercise involves classification problems.  

It is important to note that the variable `targets` (the output labels) should be a **vector**, not a matrix. Each element in the vector corresponds to the label of one input sample and can be of any type (e.g., integer, string, etc.).

Although some models may support one-hot encoded labels, others do not. Therefore, in these exercises, we will use a vector of labels, with one label per instance, rather than one-hot encoding (which is typically used in neural networks).

To prevent compatibility issues with certain model implementations, we convert all label values to `String` before passing them to the model.

Once the machine object has been created, the model can be trained using the `fit!` function as follows:

In [104]:
MLJ.fit!(mach, verbosity=0)

trained Machine; caches model-specific representations of data
  model: SVC(kernel = RadialBasis, …)
  args: 
    1:	Source @638 ⏎ Table{AbstractVector{Continuous}}
    2:	Source @315 ⏎ AbstractVector{Multiclass{3}}


This function only requires the `machine` object as an argument, since the training data has already been bound to it.
The optional argument `verbosity=0` is used to suppress output messages during training.

### Question 6.1

> ❓ What does the fact that the name of this function ends in bang (!) indicate?

`The function modifies its arguments in place`

Contrary to the Flux library, where it was necessary to write the ANN training loop, in this library the loop is already implemented, and it is called automatically when the `fit!` function is executed. Therefore, it is not necessary to write the code for the training loop.

An important aspect to consider is the layout of the data to be used.  

As shown in previous exercises, when training an Artificial Neural Network (ANN), the input samples (patterns) are arranged in **columns**, and each **row** in the input matrix represents a feature.

However, outside the scope of ANNs — and therefore for all other techniques used in this course — it is assumed that the samples are arranged in **rows**, meaning each **column** in the input matrix corresponds to a feature. This format is generally more intuitive and will be used throughout the rest of the course.


### Question 6.2

> ❓ As in the case of ANNs, a loop is necessary for training several models. Where in the code (inside or outside the loop) will you need to create the model? Which models will need to be trained several times and which ones only once? Why?

`We don’t need to retrain the deterministic models (SVMClassifier, kNNClassifier, and DTClassifier) multiple times.`
`These models always produce the same results for a given dataset and set of hyperparameters, so they only need to be created and trained once,` `typically outside the loop.`

`In contrast, the ANN starts with randomly initialized weights, and its optimization process is stochastic. Therefore, we must train it multiple` `times to estimate the expected performance of the architecture and reduce the effect of random initialization on the final results.`.


### Question 6.3

> ❓ Which condition must the matrix of inputs and the vector of desired outputs passed as an argument to this function fulfil?

`The matrix of inputs and the vector of outputs must have the same number of rows .`
`Each row in the input matrix corresponds to one observation , and each element in the output vector must represent the target value for that same observation.`

Finally, once the model has been trained, it can be used to make predictions. This is done using the `predict` function. The following is an example of how to use it:

In [10]:
testOutputs = MLJ.predict(mach, MLJ.table(test_inputs));
testOutputs

45-element CategoricalArray{String,1,UInt32}:
 "Iris-versicolor"
 "Iris-versicolor"
 "Iris-versicolor"
 "Iris-setosa"
 "Iris-versicolor"
 "Iris-versicolor"
 "Iris-virginica"
 "Iris-setosa"
 "Iris-virginica"
 "Iris-virginica"
 ⋮
 "Iris-setosa"
 "Iris-versicolor"
 "Iris-setosa"
 "Iris-versicolor"
 "Iris-setosa"
 "Iris-setosa"
 "Iris-setosa"
 "Iris-virginica"
 "Iris-setosa"

As shown, the `predict` function requires two arguments: the `machine` object and the input matrix, which must be converted to a table format.

In classification problems, the type of the prediction result depends on the model and the underlying library:

- **For SVMs**, `predict` returns a `CategoricalArray`, which can be directly compared with the ground truth labels. No post-processing is needed.
- **For Decision Trees and kNN**, `predict` returns a `UnivariateFiniteArray`, which represents a probability distribution over the possible classes.  
  To convert this into a single predicted label (so it can be compared with the true values), you can use the `mode` function to extract the most likely class.

The model being used is stored in memory as a structured object with several fields, and it can be very useful to inspect its contents.  
The `machine` object holds the model, the data, and the results of training. Therefore, you can access the trained model through the `machine`, or more directly through the variable `model`.

For example, when training an SVM, you can access one of its hyperparameters in either of the following ways:

In [11]:
model.gamma
mach.model.gamma

2.0

To inspect the learned parameters after training, MLJ provides several options.

One particularly interesting case is with SVMs, where it is useful to check which instances were selected as support vectors.  
This can be done in two ways:

In [12]:
mach.fitresult[1].SVs.indices

37-element Vector{Int32}:
  1
 12
 16
 18
 29
 34
 41
 42
  2
  3
  ⋮
 25
 26
 28
 30
 46
 47
 50
 52
 58

or using the higher-level MLJ interface:

In [13]:
fitted_params(mach)[:libsvm_model].SVs.indices

37-element Vector{Int32}:
  1
 12
 16
 18
 29
 34
 41
 42
  2
  3
  ⋮
 25
 26
 28
 30
 46
 47
 50
 52
 58

These commands return the indices of the support vectors in the training dataset.

In this notebook, the task will be to develop a single function that allows training the three different models using the MLJ library, and, in addition, artificial neural networks (ANNs) using the functions developed in previous exercises.

The training will be performed using cross-validation. For each fold, the specified model will be trained, and metrics will be computed on the test set.

As in the previous exercise, it is useful to generate a confusion matrix that reflects the distribution of instances across the test sets. In this case, it is simpler than before because the methods used are deterministic, so only one confusion matrix will be created per fold, and the final confusion matrix will be the sum of all of them.

Nevertheless, the considerations from the previous exercise still apply — in particular, that the metrics derived from this global confusion matrix may not match the metrics obtained through cross-validation.

In this exercise, you will develop a single function called `modelCrossValidation` that, in addition to training artificial neural networks (ANNs), performs cross-validation for SVMs, decision trees, and kNN.

The function should receive the following arguments:

- **`modelType::Symbol`**: This parameter indicates the type of model to train. It should take one of the following values:
  - `:ANN` — Artificial Neural Network
  - `:DoME` — DoME algorithm
  - `:SVC` — Support Vector Machine
  - `:DecisionTreeClassifier` — Decision Trees
  - `:KNeighborsClassifier` — k-Nearest Neighbors

- **`modelHyperparameters::Dict`**: A dictionary containing the model's hyperparameters. Keys may be of type `String` or `Symbol`.
  
  To check whether a hyperparameter is defined, you can use `haskey`.  
  To retrieve a value that may or may not exist in the dictionary, the `get` function is also useful.

  - **ANN (`:ANN`)**:
        The expected Hyperparameters are
        - Topology (number of hidden layers and number of neurons in each hidden layer, required) and transfer funtion in each layer. In "shallow" networks such as those used in this course, the transfer function has less impact, so a standard one, shuch as `tansig` or `logsig`, can be used.
        - Learning rate
        - Ratio of patterns used for validation
        - Number of consecutive iterations without improving the validation loss to stop the process
        - Number of times each ANN is trained.
        
### Question 6.4    
> ❓ Why should a linear transfer function not be used for neurons in the hidden layers?

`A linear transfer function should not be used in hidden layers because multiple linear transformations collapse into a single one, making the whole network equivalent to a linear model. Adding hidden layers provides no additional representational power.`
`Nonlinear activations are required to learn complex, nonlinear relationships in the data.`

  For the other models, the expected hyperparameters are:

  - **SVM (`:SVC`)**:  
    The expected hyperparameters are:
    - `C`
    - `kernel`
    - `degree`
    - `gamma`
    - `coef0`

    The `kernel` parameter should be provided as a `String` with one of the following values:
  `"linear"`, `"rbf"`, `"sigmoid"`, or `"poly"`.

    Depending on the selected kernel, some of the hyperparameters may be ignored. For example:
    - The `"poly"` kernel uses `degree`, `gamma`, and `coef0`.
    - The `"sigmoid"` kernel uses `gamma` and `coef0`.
    - The `"linear"` kernel only uses `C`.

    The `C` hyperparameter must be passed using the keyword `cost`, and the kernel must be translated to one of the predefined constants in the `LIBSVM` library:

    - `LIBSVM.Kernel.Linear`
    - `LIBSVM.Kernel.RadialBasis`
    - `LIBSVM.Kernel.Sigmoid`
    - `LIBSVM.Kernel.Polynomial`

    To avoid type errors, it is recommended to cast each value explicitly.  
    For example, to create a polynomial SVM:

  ```julia
    model = SVMClassifier(
        kernel = LIBSVM.Kernel.Polynomial,
        cost = Float64(C),
        gamma = Float64(gamma),
        degree = Int32(degree),
        coef0 = Float64(coef0)
    )
  ```

  - **Decision Tree (`:DecisionTreeClassifier`)**:

    - `max_depth`: defines the maximum depth of the tree.
    - `rng`: the random seed generator. It should be set to `Random.MersenneTwister(1)` to ensure reproducibility.

  - **k-Nearest Neighbors (`:KNeighborsClassifier`)**:
    - `n_neighbors`: the value of k, which determines the number of neighbors to consider.

- **`dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}}`**:  
  A tuple containing two elements:
  - The first is the input matrix (`X`). Unlike neural network training, there is no need to convert the data to `Float32`, since both `Float32` and `Float64` are commonly used in this library depending on the desired precision.
  - The second is the target vector (`y`), which contains the labels.

- **`crossValidationIndices::Array{Int64,1}`**:  
  This vector contains the indices used to assign each sample to a fold in the cross-validation process.

  As in the previous exercise, the fold assignment must be done **outside** the `modelCrossValidation` function.  
  This ensures that the exact same data partitioning is used when training different models, allowing fair comparisons.


The function will begin by checking whether the model to be trained is a neural network, by examining the `modelType` parameter.  
If this is the case, it will call the `ANNCrossValidation` function, passing the hyperparameters provided in `modelHyperparameters`.

Keep in mind that many of the hyperparameters for neural networks may not be defined in the dictionary.  
As mentioned earlier, the function `haskey` can be used to check whether a key is present in a `Dict`.  
Alternatively, the `get` function can be used to safely retrieve a value with a default if the key is missing.

Once the call to `ANNCrossValidation` is made, the function returns its result and exits — meaning that no further processing will occur in this case.

If a different type of model is to be trained, the logic continues similarly to the previous exercise:

- Create seven vectors to store the results of the metrics for each fold.
- Create a 2D array to accumulate the confusion matrix, initialized with zeros.

A key modification when using models from the MLJ library is to **convert the target labels to strings** before training any model.  
This helps prevent errors caused by internal type mismatches in some model implementations.

This can be done with the following simple line:

```julia
targets = string.(targets);
```

Additionally, it will be necessary to compute the vector of unique classes, just like in the previous exercise.  
This can be done with:

```julia
classes = unique(targets);
```

Once these initial steps are completed, the cross-validation loop can begin.

In each iteration, the following steps are performed:

1. Extract the training and test input matrices and the corresponding target vectors.  
   These should be of type `AbstractArray{<:Any,1}` for the targets.

2. Create the model with the specified hyperparameters.

3. For MLJ models (SVM, Decision Tree, kNN):
   - Instantiate the model using the appropriate constructor: `SVMClassifier`, `DTClassifier`, or `kNNClassifier`, depending on `modelType`.
   - Wrap the model in a `machine` with the training data.
   - Train the model using `fit!`.

4. Perform predictions on the test data using `predict`.

   - For Decision Trees and kNN, use `mode` to convert the probabilistic predictions into categorical labels:
     ```julia
     ŷ = mode.(predict(mach, MLJ.table(Xtest)))
     ```

   - For SVMs, the output of `predict` can be compared directly with the ground truth, since it returns a `CategoricalArray`.

Although the general structure of the code will be the same for the three model types, each model requires a different constructor and may require post-processing (e.g., `mode`) depending on the prediction format.

Once the predicted labels for the test set are available, the evaluation metrics and the confusion matrix should be computed using the `confusionMatrix` function.

- The metrics returned should be stored in their respective positions within the metric vectors.
- The confusion matrix obtained for each fold should be **added** to a global confusion matrix for the test set.

A key difference compared to the ANN training in the previous exercise is that these models (SVM, Decision Tree, kNN) are **deterministic**.  
Therefore, each model only needs to be trained **once per fold**, without requiring multiple executions or averaging across runs.

   ### Question 6.5
   > ❓ The other models do not have the number of times to train them as a parameter. Why? If you train several times, Which statistical properties will the results of these trainings have?

`Unlike ANNs, models such as SVMs, decision trees, and kNN are deterministic for a fixed dataset and hyperparameter configuration.
Training them multiple times with the same data and hyperparameter will produce identical results.
If they were retrained multiple times, the resulting metrics would all be identical, yielding zero variance and the results would have standard deviation = 0 (no dispersion) `

As previously described, when using techniques such as SVM, decision trees, kNN, or DoME, **one-hot encoding is not used**.  
Instead, metrics are computed using the `confusionMatrix` function developed in a previous exercise, which takes three arguments:
- The predicted labels
- The true labels
- The list of class labels

All of these must be of type `AbstractArray{<:Any,1}`.

It is important to use the version of the `confusionMatrix` function that receives the vector of classes.

### Question 6.6

> ❓ What could happen if the version that does not receive the class vector is used?

`The resulting matrix may have incorrect dimensions or inconsistent class ordering.
This happens because the function may infer the set of classes from the test data, which could be missing some classes in a given fold. As a consequence the confusion matrices from different folds would not be directly comparable or summable.
Providing the class vector ensures consistent class order and matrix dimensions across all folds`

The `modelCrossValidation` function must return the same structure as in the previous exercise: a tuple with 8 elements.

- The first **7 elements** correspond to metrics: **accuracy**, **error rate**, **recall**, **specificity**, **PPV**, **NPV**, and **F1-score**.  
  Each of these is itself a tuple with the **mean** and **standard deviation** across the folds.

- The **8th element** is the **global confusion matrix** computed on the test sets.

Once the function has been developed, it can be used to evaluate different model configurations by comparing test results across the selected metrics.  
This process does **not return a final model ready for production**, but rather identifies the best-performing model type and hyperparameter configuration.

After selecting the best configuration, the final model should be trained **from scratch** using **all available data**, without performing cross-validation.  
This training is done just once, without setting aside a test set.  
As a result, the final model is expected to perform slightly better than during cross-validation, since it benefits from more training data.

This final model is the one intended for production use, and a confusion matrix can be computed for it as well.

### Question 6.7

> ❓ In the case of using decision trees or kNN, a corresponding function is not necessary to perform the "one-against-all" strategy, why?

`Decision trees and kNN natively support multi-class classification.`
`They internally learn or compute decision boundaries that separate all classes simultaneously, without decomposing the problem into multiple binary subproblems.`
`It is not necessary to implement a one-against-all (OvA) strategy, which is  required  for algorithms like SVMs when using binary classifiers to handle multi-class tasks.`

# modelCrossValidation

In [ ]:
using MLJ, LIBSVM, DecisionTree, NearestNeighborModels, Flux
using Random, Statistics
using Flux.Losses


# --------------------------------------------------------
# Main cross-validation function
# --------------------------------------------------------
function modelCrossValidation(
        modelType::Symbol,
        modelHyperparameters::Dict,
        dataset::Tuple{AbstractArray{<:Real,2}, AbstractVector},
        crossValidationIndices::Vector{Int}
    )

    @assert modelType in [:ANN, :DoME, :SVC, :DecisionTreeClassifier, :KNeighborsClassifier] "Invalid modelType: $modelType"
    @assert isa(modelHyperparameters, Dict) "modelHyperparameters must be a Dict"
    @assert length(dataset) == 2 "dataset must be a Tuple (inputs, targets)"

    inputs, targets = dataset
    @assert size(inputs, 1) == length(targets) "inputs and targets must match"
    @assert ndims(inputs) == 2 "inputs must be 2D"
    @assert ndims(crossValidationIndices) == 1 "crossValidationIndices must be 1D"
    @assert length(crossValidationIndices) == length(targets) "CV indices must match targets"
    @assert all(crossValidationIndices .>= 1) "fold indices must be positive"
    @assert all(isfinite.(inputs)) "inputs contain NaN or Inf"

    classes = unique(targets)
    nClasses = length(classes)
    n_folds = maximum(crossValidationIndices)
    println("Executing fold cross-validation for model: $modelType")

    # Float matrix to avoid InexactError
    global_cm = zeros(Float64, nClasses, nClasses)

    for fold in 1:n_folds
        println("\n Running on Fold $fold ---")
        testIdx = findall(crossValidationIndices .== fold)
        trainIdx = setdiff(1:length(targets), testIdx)

        X_train, y_train = inputs[trainIdx, :], targets[trainIdx]
        X_test, y_test = inputs[testIdx, :], targets[testIdx]

        if modelType == :SVC
            model = SVMClassifier(
                kernel = get(modelHyperparameters, :kernel, LIBSVM.Kernel.RadialBasis),
                cost = get(modelHyperparameters, :cost, 1.0),
                gamma = get(modelHyperparameters, :gamma, 0.1)
            )
            mach = machine(model, MLJ.table(X_train), categorical(y_train))
            MLJ.fit!(mach, verbosity=0)
            y_pred = MLJ.predict(mach, MLJ.table(X_test));

        elseif modelType == :DecisionTreeClassifier
            model = DTClassifier(
                max_depth = get(modelHyperparameters, :max_depth, -1),
                min_samples_split = get(modelHyperparameters, :min_samples_split, 2)
            )

            mach = machine(model, MLJ.table(X_train), categorical(y_train))
            MLJ.fit!(mach, verbosity=0)

            # Get categorical predictions and convert to string for uniform handling
            #y_pred= MLJ.predict(mach, MLJ.table(X_test))
            y_pred = mode.(MLJ.predict(mach,  MLJ.table(X_test)))


        elseif modelType == :KNeighborsClassifier
            model = KNNClassifier(
                K = get(modelHyperparameters, :K, 3),
                algorithm = get(modelHyperparameters, :algorithm, :kdtree)
            )
            mach = machine(model, MLJ.table(X_train), categorical(y_train))
            MLJ.fit!(mach, verbosity=0)
            y_pred = mode.(MLJ.predict(mach,  MLJ.table(X_test)))

        elseif modelType == :ANN
            # --- Extract hyperparameters ---
            topology = get(modelHyperparameters, :topology, [8, 8])
            learningRate = get(modelHyperparameters, :learningRate, 0.01)
            maxEpochs = get(modelHyperparameters, :maxEpochs, 100)
            numRuns = get(modelHyperparameters, :numRuns, 3)
            validationRatio = get(modelHyperparameters, :validationRatio, 0.0)
            maxEpochsVal = get(modelHyperparameters, :maxEpochsVal, 20)

            # --- One-hot encode targets ---
            classes = unique(y_train)
            y_train_oh = oneHotEncoding(y_train, classes)
            y_test_oh = oneHotEncoding(y_test, classes)

            nClasses = length(classes)
            println("Training ANN model with $(nClasses) classes, topology = $topology")

            # --- Prepare storage for aggregate predictions ---
            all_preds = zeros(Bool, length(y_test))
            
            for run in 1:numRuns
                println(" ANN training run $run / $numRuns...")

                # Optional hold-out split for validation if validationRatio > 0
                if validationRatio > 0
                    N = size(X_train, 1)
                    (train_idx, val_idx) = holdOut(N, validationRatio)
                    trainInputs = X_train[train_idx, :]
                    valInputs = X_train[val_idx, :]
                    trainTargets = y_train_oh[train_idx, :]
                    valTargets = y_train_oh[val_idx, :]
                else
                    trainInputs = X_train
                    valInputs = Array{eltype(X_train), 2}(undef, 0, 0)
                    trainTargets = y_train_oh
                    valTargets = falses(0, 0)
                end

                # --- Train the ANN ---
                finalANN, trainLoss, valLoss, testLoss = trainClassANN(
                    topology,
                    (trainInputs, trainTargets),
                    validationDataset = (valInputs, valTargets),
                    testDataset = (X_test, y_test_oh),
                    learningRate = learningRate,
                    maxEpochs = maxEpochs,
                    maxEpochsVal = maxEpochsVal,
                    showText = false
                )

                # --- Make predictions ---
                testOutputs = finalANN(X_test')
                testPredictions = classifyOutputs(testOutputs')
                predictedLabels = Flux.onecold(testPredictions', classes)
                all_preds .|= (predictedLabels .== y_test)
            end



            # --- Final predictions ---
            y_pred = all_preds
        end

        # Compute confusion matrix
        cm, acc, err, _, _, _, _, _ = confusionMatrix(y_pred, y_test)
        println("\n=== ANN Confusion Matrix ===")
        println(round.(Int, cm))  # optional integer di
        global_cm .+= cm
        println("Fold $fold accuracy = $(round(acc*100, digits=2))%")
    end

    println("\n=== Global Confusion Matrix ===")
    println(round.(Int, global_cm))  # optional integer display

    return global_cm
end



using MLJ, LIBSVM, DecisionTree, NearestNeighborModels, Flux
using Random, Statistics

# --------------------------------------------------------
#  Load and prepare the Iris dataset
# --------------------------------------------------------
dataset = readdlm("./data/iris.data", ',')  # or use built-in Iris
inputs = convert(Array{Float32,2}, dataset[:,1:4])
targets = dataset[:,5]

# Encode categorical targets
classes = unique(targets)
targets_cat = categorical(targets)

println("Classes: ", classes)
println("Dataset size: ", size(inputs))

# --------------------------------------------------------
# Create 5-fold cross-validation indices
# --------------------------------------------------------
n_samples = length(targets)
n_folds = 5
fold_indices = repeat(1:n_folds, inner=ceil(Int, n_samples / n_folds))[1:n_samples]
fold_indices = shuffle(fold_indices)

println("Cross-validation indices: ", fold_indices[1:10])

# # --------------------------------------------------------
# #  Train SVM (SVC)
# # --------------------------------------------------------
# svm_hyper = Dict(
#     :kernel => LIBSVM.Kernel.RadialBasis,
#     :cost => 1.0,
#     :gamma => 0.2
# )

# println("\n Running SVM Cross-Validation...")
# cm_svm = modelCrossValidation(:SVC, svm_hyper, (inputs, targets_cat), fold_indices)
# println("SVM confusion matrix:\n", cm_svm)


# # --------------------------------------------------------
# #  Train Decision Tree
# # --------------------------------------------------------
tree_hyper = Dict(
    :max_depth => 5,
    :min_samples_split => 2
)

println("\n Running Decision Tree Cross-Validation...")
cm_tree = modelCrossValidation(:DecisionTreeClassifier, tree_hyper, (inputs, targets_cat), fold_indices)
println("Decision Tree confusion matrix:\n", cm_tree)


# --------------------------------------------------------
# Train K-Nearest Neighbors
# --------------------------------------------------------
# knn_hyper = Dict(
#     :K => 3,
#     :algorithm => :kdtree
# )

# println("\n Running KNN Cross-Validation...")
# cm_knn = modelCrossValidation(:KNeighborsClassifier, knn_hyper, (inputs, targets_cat), fold_indices)
# println("KNN confusion matrix:\n", cm_knn) 


# # --------------------------------------------------------
# # Train Artificial Neural Network
# # --------------------------------------------------------
# ann_hyper = Dict(
#     :topology => [8, 8],
#     :learningRate => 0.01,
#     :maxEpochs => 1,
#     :numRuns => 3
# )

# println("\n Running ANN Cross-Validation...")
# cm_ann = modelCrossValidation(:ANN, ann_hyper, (inputs, targets_cat), fold_indices)
# println("ANN confusion matrix:\n", cm_ann)


In [95]:
using MLJ, LIBSVM, DecisionTree, NearestNeighborModels, Flux
using Statistics

function modelCrossValidation(
    modelType::Symbol,
    modelHyperparameters::Dict,
    dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
    crossValidationIndices::Vector{Int64}
)
    inputs, targets = dataset
    n_folds = maximum(crossValidationIndices)
    classes = unique(targets)
    n_classes = length(classes)

    println("\nExecuting fold cross-validation for model: $modelType")

    # Initialize metrics
    accs = Float64[]; errs = Float64[]; recalls = Float64[]
    specs = Float64[]; ppvs = Float64[]; npvs = Float64[]; f1s = Float64[]
    global_cm = zeros(Float32, n_classes, n_classes)

    if modelType == :ANN
        # ANN hyperparameters
        topology = get(modelHyperparameters, :topology, [10,10])
        numExecutions = get(modelHyperparameters, :numExecutions, 5)
        maxEpochs = get(modelHyperparameters, :maxEpochs, 100)
        learningRate = get(modelHyperparameters, :learningRate, 0.01)
        validationRatio = get(modelHyperparameters, :validationRatio, 0.1)
        maxEpochsVal = get(modelHyperparameters, :maxEpochsVal, 20)
        transferFunctions = get(modelHyperparameters, :transferFunctions, fill(σ, length(topology)))

        # Call ANNCrossValidation on the whole dataset
        ann_metrics = ANNCrossValidation(
            topology,
            (convert(Array{Float32,2}, inputs), targets),
            crossValidationIndices;
            numExecutions=numExecutions,
            transferFunctions=transferFunctions,
            maxEpochs=maxEpochs,
            learningRate=learningRate,
            validationRatio=validationRatio,
            maxEpochsVal=maxEpochsVal
        )

        # ann_metrics returns 8-tuple including global confusion matrix
        return ann_metrics

    else
        # Other MLJ models (SVM, DecisionTree, KNN)
        for fold in 1:n_folds
            println("\n Running on Fold $fold ---")
            testIdx = findall(crossValidationIndices .== fold)
            trainIdx = setdiff(1:length(targets), testIdx)

            X_train, y_train = inputs[trainIdx, :], targets[trainIdx]
            X_test,  y_test  = inputs[testIdx,  :], targets[testIdx]


            if modelType == :SVC
                model = SVMClassifier(
                    kernel = get(modelHyperparameters, :kernel, LIBSVM.Kernel.RadialBasis),
                    cost   = get(modelHyperparameters, :cost, 1.0),
                    gamma  = get(modelHyperparameters, :gamma, 0.1)
                )
                mach = machine(model, MLJ.table(X_train), categorical(y_train))
                MLJ.fit!(mach, verbosity=0)
                y_pred = MLJ.predict(mach, MLJ.table(X_test))
                accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(string.(y_pred), string.(y_test))
                global_cm .+= Float32.(cm)

            elseif modelType == :DTClassifier
                model = DTClassifier(
                    max_depth = get(modelHyperparameters, :max_depth, -1),
                    min_samples_split = get(modelHyperparameters, :min_samples_split, 2)
                )

                mach = machine(model, MLJ.table(X_train), categorical(y_train))
                MLJ.fit!(mach, verbosity=0)

                # Get categorical predictions and convert to string for uniform handling
                #y_pred= MLJ.predict(mach, MLJ.table(X_test))
                y_pred = mode.(MLJ.predict(mach,  MLJ.table(X_test)))

                accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(y_pred, y_test)
                classes = unique(targets)
                nClasses = length(classes)
                global_cm = zeros(Float64, nClasses, nClasses)
                # cm is integer matrix; convert to Float32 for accumulation
                global_cm .+= Float32.(cm)

            elseif modelType == :KNeighborsClassifier
                model = KNNClassifier(
                    K = get(modelHyperparameters, :K, 3),
                    algorithm = get(modelHyperparameters, :algorithm, :kdtree)
                )

                mach = machine(model, MLJ.table(X_train), categorical(y_train))
                MLJ.fit!(mach, verbosity=0)
                y_pred = mode.(MLJ.predict(mach,  MLJ.table(X_test)))
                accuracy, error_rate, recall, spec, ppv, npv, f1, cm =confusionMatrix(string.(y_pred), string.(y_test))
                global_cm .+= Float32.(cm)

            else
                error("Unknown model type: $modelType")
            end



            # Store metrics per fold
            push!(accs, accuracy); push!(errs, error_rate); push!(recalls, recall)
            push!(specs, spec); push!(ppvs, ppv); push!(npvs, npv); push!(f1s, f1)

            
            println("Accuracy (mean, std): ", acc)
            println("err (mean, std): ", err)
            println("recall (mean, std): ", recall)
            println("spec (mean, std): ", spec)
            println("ppv (mean, std): ", ppv)
            println("npv (mean, std): ", npv)
            println("f1 (mean, std): ", f1)
            println("Confusion matrix:\n", cm)

        end

        # Return mean/std metrics + global confusion matrix
        metrics_summary = (
            (mean(accs), std(accs)),
            (mean(errs), std(errs)),
            (mean(recalls), std(recalls)),
            (mean(specs), std(specs)),
            (mean(ppvs), std(ppvs)),
            (mean(npvs), std(npvs)),
            (mean(f1s), std(f1s)),
            global_cm
        )
        return metrics_summary
    end
end


# --------------------------------------------------------
# Load and prepare the Iris dataset
# --------------------------------------------------------
dataset = readdlm("./data/iris.data", ',')  # or use built-in Iris
inputs = convert(Array{Float32,2}, dataset[:,1:4])
targets = dataset[:,5]

# Encode categorical targets
classes = unique(targets)
targets_cat = categorical(targets)

println("Classes: ", classes)
println("Dataset size: ", size(inputs))

# --------------------------------------------------------
# Create 5-fold cross-validation indices
# --------------------------------------------------------
n_samples = length(targets)
n_folds = 5
fold_indices = repeat(1:n_folds, inner=ceil(Int, n_samples / n_folds))[1:n_samples]
fold_indices = shuffle(fold_indices)

println("Cross-validation indices: ", fold_indices[1:10])

# # --------------------------------------------------------
# #  Train SVM (SVC) WORKS!
# # --------------------------------------------------------
svm_hyper = Dict(
    :kernel => LIBSVM.Kernel.RadialBasis,
    :cost => 1.0,
    :gamma => 0.2
)

println(repeat("=", 20), "Training SVM with Cross-Validation...", repeat("=", 20))
(acc, err, recall, spec, ppv, npv, f1, cm) = modelCrossValidation(:SVC, svm_hyper, (inputs, targets_cat), fold_indices)
# println("Accuracy (mean, std): ", acc)
# println("err (mean, std): ", err)
# println("recall (mean, std): ", recall)
# println("spec (mean, std): ", spec)
# println("ppv (mean, std): ", ppv)
# println("npv (mean, std): ", npv)
# println("f1 (mean, std): ", f1)
println("SVM Confusion matrix:\n", cm)

# --------------------------------------------------------
#  Train Decision Tree DOES NOT WORK
# --------------------------------------------------------
tree_hyper = Dict(
    :max_depth => 5,
    :min_samples_split => 2
)

println(repeat("=", 20), "Training Decision Tree with Cross-Validation...", repeat("=", 20))
(acc, err, recall, spec, ppv, npv, f1, cm) = modelCrossValidation(:DTClassifier, tree_hyper, (inputs, targets_cat), fold_indices)
println("Decision Tree confusion matrix:\n", cm)

# --------------------------------------------------------
# Train K-Nearest Neighbors WORKS
# --------------------------------------------------------
knn_hyper = Dict(
    :K => 3,
    :algorithm => :kdtree
)


println(repeat("=", 20), "Training KNN with Cross-Validation...", repeat("=", 20))

(acc, err, recall, spec, ppv, npv, f1, cm) = modelCrossValidation(:KNeighborsClassifier, knn_hyper, (inputs, targets_cat), fold_indices)
# println("Accuracy (mean, std): ", acc)
# println("err (mean, std): ", err)
# println("recall (mean, std): ", recall)
# println("spec (mean, std): ", spec)
# println("ppv (mean, std): ", ppv)
# println("npv (mean, std): ", npv)
# println("f1 (mean, std): ", f1)
println("KNN Confusion matrix:\n", cm)


# --------------------------------------------------------
# Train Artificial Neural Network
# --------------------------------------------------------
println(repeat("=", 20), "Train Artificial Neural Network", repeat("=", 20)) 
ann_hyper = Dict(
    :topology => [8, 8],
    :learningRate => 0.01,
    :maxEpochs => 1,
    :numRuns => 3
)

(acc, err, recall, spec, ppv, npv, f1, cm) =
    modelCrossValidation(:ANN, ann_hyper, (inputs, targets_cat), fold_indices)

# println("Accuracy (mean, std): ", acc)
# println("err (mean, std): ", err)
# println("recall (mean, std): ", recall)
# println("spec (mean, std): ", spec)
# println("ppv (mean, std): ", ppv)
# println("npv (mean, std): ", npv)
# println("f1 (mean, std): ", f1)
println("ANN Confusion matrix:\n", cm)


Classes: Any["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
Dataset size: (150, 4)
Cross-validation indices: [2, 4, 4, 4, 3, 2, 1, 1, 3, 5]
====================Training SVM with Cross-Validation...====================

Executing fold cross-validation for model: SVC

 Running on Fold 1 ---
Accuracy (mean, std): (0.89066666f0, 0.09575431f0)
err (mean, std): (0.10933334f0, 0.095754325f0)
recall (mean, std): 1.0
spec (mean, std): 1.0
ppv (mean, std): 1.0
npv (mean, std): 1.0
f1 (mean, std): 1.0
Confusion matrix:
[8 0 0; 0 12 0; 0 0 10]

 Running on Fold 2 ---
Accuracy (mean, std): (0.89066666f0, 0.09575431f0)
err (mean, std): (0.10933334f0, 0.095754325f0)
recall (mean, std): 0.9333333333333333
spec (mean, std): 0.9634920634920635
ppv (mean, std): 0.9333333333333333
npv (mean, std): 0.9634920634920635
f1 (mean, std): 0.9333333333333333
Confusion matrix:
[9 0 0; 0 11 1; 0 1 8]

 Running on Fold 3 ---
Accuracy (mean, std): (0.89066666f0, 0.09575431f0)
err (mean, std): (0.10933334f0, 0.09

In [ ]:
function modelCrossValidation(
    modelType::Symbol,
    modelHyperparameters::Dict,
    dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
    crossValidationIndices::Vector{Int64}
)
    # Input validation
    @assert modelType in [:ANN, :SVC, :DecisionTreeClassifier, :KNeighborsClassifier] "Invalid modelType: $modelType"
    @assert length(dataset) == 2 "dataset must be (inputs, targets)"
    inputs, targets = dataset
    @assert size(inputs, 1) == length(targets) "inputs and targets mismatch"
    @assert length(crossValidationIndices) == length(targets) "Cross-validation indices must match targets length"

    n_folds = maximum(crossValidationIndices)
    classes = unique(targets)
    n_classes = length(classes)

    println("\nExecuting fold cross-validation for model: $modelType")

    # Use Float32 for ANN (average matrices may be floats)
    global_cm = zeros(Float32, n_classes, n_classes)

    # metric containers
    accs = Float64[]; errs = Float64[]; recalls = Float64[]
    specs = Float64[]; ppvs = Float64[]; npvs = Float64[]; f1s = Float64[]

    for fold in 1:n_folds
        println("\n Running on Fold $fold ---")
        testIdx = findall(crossValidationIndices .== fold)
        trainIdx = setdiff(1:length(targets), testIdx)

        X_train, y_train = inputs[trainIdx, :], targets[trainIdx]
        X_test,  y_test  = inputs[testIdx,  :], targets[testIdx]

        X_train_tbl = MLJ.table(X_train)
        X_test_tbl  = MLJ.table(X_test)

        if modelType == :SVC
            model = SVMClassifier(
                kernel = get(modelHyperparameters, :kernel, LIBSVM.Kernel.RadialBasis),
                cost   = get(modelHyperparameters, :cost, 1.0),
                gamma  = get(modelHyperparameters, :gamma, 0.1)
            )
            mach = machine(model, X_train_tbl, categorical(y_train))
            fit!(mach, verbosity=0)
            y_pred = MLJ.predict_mode(mach, X_test_tbl)

        elseif modelType == :DecisionTreeClassifier
            model = DecisionTreeClassifier(
                max_depth = get(modelHyperparameters, :max_depth, -1),
                min_samples_split = get(modelHyperparameters, :min_samples_split, 2)
            )
            mach = machine(model, X_train_tbl, categorical(y_train))
            fit!(mach, verbosity=0)
            y_pred = MLJ.predict_mode(mach, X_test_tbl)

        elseif modelType == :KNeighborsClassifier
            model = KNNClassifier(
                K = get(modelHyperparameters, :K, 3),
                algorithm = get(modelHyperparameters, :algorithm, :kdtree)
            )
            mach = machine(model, X_train_tbl, categorical(y_train))
            fit!(mach, verbosity=0)
            y_pred = MLJ.predict_mode(mach, X_test_tbl)

        elseif modelType == :ANN
            # ANN hyperparameters
            topology = get(modelHyperparameters, :topology, [10, 10])
            maxEpochs = get(modelHyperparameters, :maxEpochs, 500)
            learningRate = get(modelHyperparameters, :learningRate, 0.01)
            transferFunctions = get(modelHyperparameters, :transferFunctions, fill(σ, length(topology)))

            # Convert inputs to Float32
            X_train_f32 = convert(Array{Float32,2}, X_train)
            X_test_f32  = convert(Array{Float32,2}, X_test)

            # Train ANN on this fold
            ann_model = trainClassANN(
                X_train_f32, y_train;
                topology=topology,
                maxEpochs=maxEpochs,
                learningRate=learningRate,
                transferFunctions=transferFunctions
            )

            # Predict on test set
            y_pred = predictClassANN(ann_model, X_test_f32, classes)
        else
            error("Unknown model type: $modelType")
        end

        # --- metrics ---
        cm = confusionMatrix(y_test, y_pred)
        global_cm .+= Float32.(cm)  # accumulate as Float32

        metrics = evaluateClassifier(cm)
        acc, err, recall, spec, ppv, npv, f1 = metrics

        push!(accs, acc)
        push!(errs, err)
        push!(recalls, recall)
        push!(specs, spec)
        push!(ppvs, ppv)
        push!(npvs, npv)
        push!(f1s, f1)

        println("Fold $fold accuracy = $(round(acc * 100, digits=2))%")
    end

    metrics_summary = [
        (mean(accs), std(accs)),
        (mean(errs), std(errs)),
        (mean(recalls), std(recalls)),
        (mean(specs), std(specs)),
        (mean(ppvs), std(ppvs)),
        (mean(npvs), std(npvs)),
        (mean(f1s), std(f1s))
    ]

    println("\n=== Global Confusion Matrix ===")
    println(global_cm)

    return (metrics_summary..., global_cm)
end

# --------------------------------------------------------
# Train Artificial Neural Network
# --------------------------------------------------------
ann_hyper = Dict(
    :topology => [8, 8],
    :learningRate => 0.01,
    :maxEpochs => 1,
    :numRuns => 3
)

println("\n Running ANN Cross-Validation...")
cm_ann = modelCrossValidation(:ANN, ann_hyper, (inputs, targets_cat), fold_indices)
println("ANN confusion matrix:\n", cm_ann)

### Learn Julia

#### Symbols and Dictionaries in Julia
One Julia type that is important for this exercise is the `Symbol` type. An object of this type can be any symbol you want, simply by typing its name after a colon (":"). In this practice, you can use it to indicate which model you want to train, for example, in the `modelCrossValidation` function, symbols will be used to indicate which model to train:

```julia
:KNeighborsClassifier, :SVC, :DecisionTreeClassifier, :ANN
```

#### Passing Model-Specific Parameters
This function will also require model-specific parameters to be passed.
The recommended way to do this is to define a variable of type Dict, which works similarly to Python dictionaries.

For instance, to define the hyperparameters for an artificial neural network (ANN):

```julia
  modelHyperparameters = Dict(
      "topology" => [5, 3],
      "learningRate" => 0.01,
      "validationRatio" => 0.2,
      "numExecutions" => 50,
      "maxEpochs" => 1000,
      "maxEpochsVal" => 6
  )
```

Another way to define the same dictionary:

```julia
  modelHyperparameters = Dict()
  modelHyperparameters["topology"] = topology
  modelHyperparameters["learningRate"] = learningRate
  modelHyperparameters["validationRatio"] = validationRatio
  modelHyperparameters["numExecutions"] = numRepetitionsANNTraining
  modelHyperparameters["maxEpochs"] = numMaxEpochs
  modelHyperparameters["maxEpochsVal"] = maxEpochsVal
```
To access a value, simply use:
```julia
  modelHyperparameters["topology"]
```
#### Example for SVM Parameters
You can also define hyperparameters for other models similarly.
For example, for an SVM:
```julia
  modelHyperparameters = Dict("C" => 1, "kernel" => "rbf", "gamma" => 2)
```
Or using the alternative form:

```julia
  modelHyperparameters = Dict()
  modelHyperparameters["C"] = 1
  modelHyperparameters["kernel"] = "rbf"
  modelHyperparameters["gamma"] = 2
```
Other kernels may require different parameters, such as `degree` and `coef0`.

When building the SVM model inside the function, you might write:
```julia
  if modelHyperparameters["kernel"] == "rbf"
    model = SVMClassifier(
        kernel = LIBSVM.Kernel.RadialBasis,
        cost = Float64(modelHyperparameters["C"]),
        gamma = Float64(modelHyperparameters["gamma"])
    )
```

You can apply a similar strategy for decision trees, kNN, and DoME models.

In the examples above, the dictionary keys are `String`, but you may also use `Symbol` keys interchangeably.
For example:
```julia
  modelHyperparameters = Dict(:C => 1, :kernel => "rbf", :gamma => 2)
```

Another type of Julia that may be interesting for this assignment is the `Symbol` type. An object of this type can be any symbol you want, simply by typing its name after a colon (":"). In this practice, you can use it to indicate which model you want to train, for example `:ANN`, `:SVM`, `:DecisionTree` or `:kNN`.